# Lección 15.2 — Monitoreo: deriva de datos y de concepto

> **Módulo 15 · ML en producción y responsabilidad** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 15.1; Lección 4.2 (métricas de clasificación) y Lección 4.3 (AUC)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir **deriva de datos** (*data drift*), **deriva de concepto** (*concept drift*) y cambio en la proporción de clases.
- Detectar deriva de datos con **pruebas estadísticas** (Kolmogórov-Smirnov, chi-cuadrado) y con el **índice PSI**.
- Evitar las dos trampas clásicas: **muestras enormes** que alarman por nada y **muchas pruebas** que generan falsas alarmas.
- Diseñar un **tablero de monitoreo** mensual y explicar por qué la deriva de concepto solo se ve con etiquetas.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el GPS con el mapa viejo 🗺️

Tu GPS aprendió las calles de la ciudad hace dos años. Desde entonces:
- abrieron **barrios nuevos** por donde el GPS nunca pasó → las **entradas** cambiaron;
- una calle que antes era rápida ahora tiene **obras** y es lentísima → la **relación** "esta ruta = rápida" cambió.

El GPS no da error: sigue recomendando rutas con total seguridad. Simplemente, **cada vez acierta menos**.

Un modelo en producción vive lo mismo. Fue entrenado con una "foto" del pasado, y el mundo sigue moviéndose. A ese movimiento se le llama **deriva** (*drift*), y vigilarla es el **monitoreo**.

### ✋ Ejercicio 1 — Calentamiento
Un modelo que predice la demanda de helados se entrenó con datos de 2019. ¿Qué eventos de los años siguientes podrían hacer que falle? Piensa en al menos dos de tipo distinto.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Por ejemplo:
- **La pandemia de 2020**: cierres, gente en casa, ventas por entrega a domicilio. Las relaciones de siempre ("fin de semana soleado = muchas ventas en el parque") dejaron de funcionar.
- **Inflación**: los precios en pesos subieron; un "ticket alto" de 2019 hoy es normal.
- **Olas de calor** más frecuentes, una **heladería nueva** en la esquina, un **cambio de menú**.

Fíjate en que algunos cambian **lo que entra** al modelo (precios, temperaturas) y otros cambian **cómo se relaciona** lo que entra con lo que sale (la pandemia). Esa es justo la distinción de la siguiente sección.
</details>

---
## 2. 📖 El concepto: tres tipos de deriva

Un modelo aprende la relación entre las entradas **X** y la salida **y**. Cualquiera de las dos partes puede cambiar:

| Tipo | Qué cambia | Ejemplo (modelo de cancelación de clientes) | ¿Se ve sin etiquetas? |
|---|---|---|---|
| **Deriva de datos** (*data drift*, *covariate shift*) | La distribución de las entradas X | Por la inflación, los cargos mensuales suben un 20 % | ✅ Sí: basta mirar X |
| **Deriva de concepto** (*concept drift*) | La relación entre X e y | Un competidor ofrece romper contratos gratis: el contrato anual ya no retiene | ❌ No: X se ve igual; hace falta saber qué pasó (y) |
| **Cambio en la proporción de clases** (*label shift*) | Qué tan frecuente es cada clase | Una crisis hace que se vaya el doble de clientes | Parcialmente: se nota en las predicciones |

> 💡 La deriva de datos **no siempre** daña al modelo (si la relación se mantiene y el modelo sabe extrapolar), y la deriva de concepto **siempre** lo daña. Pero la de datos es la más fácil de detectar.

### ✋ Ejercicio 2 — Clasifica la deriva
Un banco tiene un modelo de fraude con tarjetas. ¿Qué tipo de deriva es cada caso?
1. El banco lanza su tarjeta en otro país: llegan muchas compras en otra moneda y en comercios nuevos.
2. Los estafadores aprenden a imitar el patrón de compras de los clientes reales: las mismas compras que antes eran "normales" ahora a veces son fraude.
3. Durante el Buen Fin / Black Friday se triplican las compras grandes en tiendas en línea.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Deriva de datos**: cambian las entradas (moneda, comercios).
2. **Deriva de concepto**: las mismas entradas ahora significan otra cosa. Es típica del fraude, porque el "adversario" se adapta al modelo (Lección 15.5).
3. **Deriva de datos**, temporal y estacional. No hay que entrar en pánico: el monitoreo debe comparar con lo que es normal **para esa época** (o con el Buen Fin del año pasado).
</details>

---
## 3. 📖 Cómo se detecta la deriva de datos

Idea: comparar la distribución de cada variable en una **ventana de referencia** (los datos de entrenamiento) con una **ventana actual** (lo que llegó este mes).

▶️ Empecemos generando los clientes de entrenamiento y un mes de producción en el que **los cargos subieron un 20 %** por la inflación.

In [ ]:
def generar_clientes(n, semilla, inflacion=1.0, efecto_contrato=-1.8, prob_planes=(.5, .3, .2)):
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "meses_cliente":    rng.integers(1, 72, n),
        "llamadas_soporte": rng.poisson(1.5, n),
        "cargo_mensual":    rng.integers(200, 1_500, n) * inflacion,
        "contrato_anual":   rng.integers(0, 2, n),
        "plan":             rng.choice(["básico", "plus", "premium"], n, p=prob_planes),
    })
    cargo_real = df.cargo_mensual / inflacion            # al cliente le importa el precio "real"
    riesgo = (-1.5 + 0.9 * df.llamadas_soporte - 0.05 * df.meses_cliente + 0.002 * cargo_real
              + efecto_contrato * df.contrato_anual + 0.4 * (df.plan == "básico"))
    df["cancelo"] = (rng.random(n) < 1 / (1 + np.exp(-riesgo))).astype(int)
    return df

referencia = generar_clientes(5_000, semilla=0)                  # datos de entrenamiento
mes_inflacion = generar_clientes(2_000, semilla=1, inflacion=1.2)

fig, ax = plt.subplots()
ax.hist(referencia.cargo_mensual, bins=30, alpha=.5, density=True, label="Referencia (entrenamiento)")
ax.hist(mes_inflacion.cargo_mensual, bins=30, alpha=.5, density=True, label="Mes actual (+20 %)")
ax.set_xlabel("cargo_mensual"); ax.legend(); plt.show()

### 📖 Prueba de Kolmogórov-Smirnov (KS)
Para variables **numéricas**, la prueba KS compara dos muestras y responde: *"¿es creíble que vengan de la misma distribución?"*.
- **Estadístico KS** (0 a 1): la mayor distancia entre las dos distribuciones acumuladas. 0 = idénticas.
- **Valor p**: si es muy pequeño (por ejemplo, < 0.05), es poco creíble que la diferencia sea solo azar → **alarma**.

### 🔮 Ejercicio 3 — ¿Quién deriva?
Aplicaremos KS a `cargo_mensual` y a `meses_cliente`. ¿Cuál crees que tendrá un valor p diminuto y cuál uno "normal"?

✍️ *Mi predicción:*

In [ ]:
for col in ["cargo_mensual", "meses_cliente", "llamadas_soporte"]:
    ks = ks_2samp(referencia[col], mes_inflacion[col])
    print(f"{col:<17} KS = {ks.statistic:.3f} · valor p = {ks.pvalue:.2g}  {'🚨 deriva' if ks.pvalue < 0.05 else '✅ estable'}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `cargo_mensual`: KS ≈ 0.21 y valor p **diminuto** (≈ 10⁻⁵³) → 🚨 deriva (la subimos un 20 %).
- `meses_cliente` y `llamadas_soporte`: KS ≈ 0.02 y valor p ≈ 0.75 → ✅ estables. (`llamadas_soporte` es una variable de conteo, con muchos empates; para ellas KS es algo conservadora y a veces se prefiere chi-cuadrado.)

¡Ojo! Detectar deriva en `cargo_mensual` **no dice** si el modelo empeoró. Solo dice que **ahora ve clientes distintos**. Es una señal para investigar.
</details>

### 📖 Para variables categóricas: chi-cuadrado
Para categorías (como el `plan`) se compara la tabla de frecuencias con una prueba **chi-cuadrado** (χ²). Supón que marketing lanzó una promoción del plan premium:

In [ ]:
mes_promo = generar_clientes(2_000, semilla=2, prob_planes=(.35, .3, .35))   # más premium

tabla = pd.DataFrame({"referencia": referencia.plan.value_counts(), "mes actual": mes_promo.plan.value_counts()})
print(tabla.assign(**{"% ref": (tabla.referencia / tabla.referencia.sum() * 100).round(1),
                      "% actual": (tabla["mes actual"] / tabla["mes actual"].sum() * 100).round(1)}))
chi2, p, _, _ = chi2_contingency(tabla.values)
print(f"\nχ² = {chi2:.1f} · valor p = {p:.2g}")

### 📖 El índice de estabilidad poblacional (PSI)
Los valores p tienen un problema que verás enseguida. Por eso en la industria (sobre todo en banca) se usa mucho el **PSI** (*Population Stability Index*), que mide **cuánto** cambió la distribución, no si el cambio es "estadísticamente significativo":

1. Divide la variable en 10 tramos usando los cuantiles de la **referencia** (10 % de los datos en cada tramo).
2. Mira qué % de los datos actuales cae en cada tramo.
3. Suma, tramo a tramo: (% actual − % referencia) × ln(% actual / % referencia).

| PSI | Interpretación habitual |
|---|---|
| < 0.1 | Sin cambio relevante |
| 0.1 – 0.25 | Cambio moderado: vigilar |
| > 0.25 | Cambio grande: investigar / probablemente reentrenar |

In [ ]:
def psi(referencia, actual, tramos=10):
    cortes = np.unique(np.quantile(referencia, np.linspace(0, 1, tramos + 1)))
    cortes[0], cortes[-1] = -np.inf, np.inf
    p_ref = np.histogram(referencia, cortes)[0] / len(referencia)
    p_act = np.histogram(actual, cortes)[0] / len(actual)
    p_ref, p_act = np.clip(p_ref, 1e-4, None), np.clip(p_act, 1e-4, None)   # evita log(0)
    return float(np.sum((p_act - p_ref) * np.log(p_act / p_ref)))

for col in ["cargo_mensual", "meses_cliente"]:
    print(f"{col:<15} PSI = {psi(referencia[col], mes_inflacion[col]):.3f}")

### ✋ Ejercicio 4 — Lee el PSI
1. ¿En qué zona de la tabla cae `cargo_mensual` con un 20 % de inflación?
2. ✏️ Cambia la inflación a 1.05 (5 %) y a 1.5 (50 %) en la celda de abajo. ¿A partir de qué inflación pasa a "investigar"?

✍️ *Mis respuestas:*

In [ ]:
INFLACION = 1.2     # ✏️ prueba 1.05, 1.2, 1.5
mes = generar_clientes(2_000, semilla=1, inflacion=INFLACION)
print(f"Inflación +{INFLACION - 1:.0%} → PSI(cargo_mensual) = {psi(referencia.cargo_mensual, mes.cargo_mensual):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con 20 % de inflación, PSI ≈ **0.26**: ya está en la zona de **investigar** (> 0.25).
2. Con 5 %, PSI ≈ **0.04** (sin cambio relevante); con 50 %, PSI ≈ **0.92**, enorme. El umbral de "investigar" se cruza alrededor del **20 %** de inflación.

El PSI se lee como una **regla de tres**: no dice "hay o no hay diferencia", sino "**qué tan grande** es".
</details>

---
## 4. 📖 ¿Qué pasa si…? Dos trampas de los valores p

### Trampa 1: con muchísimos datos, todo es "significativo"
Con millones de filas, la prueba KS detecta diferencias **minúsculas** que no le importan a nadie.

### 🔮 Ejercicio 5 — Un 1 % de inflación
Comparamos referencia y actual con apenas **1 %** de inflación, primero con 2,000 clientes por mes y luego con 200,000. ¿Qué dirá KS en cada caso? ¿Y el PSI?

✍️ *Mi predicción:*

In [ ]:
for n in [2_000, 200_000]:
    ref_grande = generar_clientes(n, semilla=10)
    act_grande = generar_clientes(n, semilla=11, inflacion=1.01)
    ks = ks_2samp(ref_grande.cargo_mensual, act_grande.cargo_mensual)
    print(f"n = {n:>7,} → valor p KS = {ks.pvalue:.2g} {'🚨' if ks.pvalue < 0.05 else '✅'} · "
          f"PSI = {psi(ref_grande.cargo_mensual, act_grande.cargo_mensual):.4f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Con **2,000** clientes: valor p grande → ✅ sin alarma.
- Con **200,000**: valor p **diminuto** → 🚨 ¡alarma por un 1 % de inflación!
- El **PSI** queda prácticamente en 0 en ambos casos: el cambio es real pero **irrelevante**.

Un valor p pequeño significa "estoy **seguro** de que hay alguna diferencia", no "la diferencia es **grande**". Con muchos datos conviene mirar el **tamaño del efecto** (el estadístico KS, el PSI) además del valor p.
</details>

### Trampa 2: muchas variables, muchas falsas alarmas
Si monitoreas 50 variables cada mes con un umbral de p < 0.05, **por puro azar** esperas que alrededor de 1 de cada 20 dé alarma aunque nada haya cambiado.

### 🔮 Ejercicio 6 — 50 variables sin deriva
Generamos 50 variables que **no cambian en absoluto** (misma distribución en referencia y actual). ¿Cuántas alarmas esperas con p < 0.05? ¿Y con la corrección de **Bonferroni** (dividir 0.05 entre el número de pruebas)?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(3)
ref_50 = rng.normal(size=(2_000, 50))
act_50 = rng.normal(size=(2_000, 50))          # ¡exactamente la misma distribución!
valores_p = np.array([ks_2samp(ref_50[:, j], act_50[:, j]).pvalue for j in range(50)])

print(f"Alarmas con p < 0.05            : {(valores_p < 0.05).sum()} de 50")
print(f"Alarmas con Bonferroni (p < {0.05 / 50:.3f}): {(valores_p < 0.05 / 50).sum()} de 50")

<details><summary><b>👀 Ver respuesta</b></summary>

Con p < 0.05 salen **4 alarmas falsas** de 50 (se esperaban unas 50 × 0.05 = 2.5; el resto es azar). Con Bonferroni, **ninguna**.

En un tablero real, 5 alarmas falsas cada mes hacen que el equipo **deje de hacer caso** a las alertas ("fatiga de alertas"). Soluciones: corregir por múltiples pruebas, usar umbrales de PSI, pedir que la alarma se repita dos meses seguidos, o vigilar con prioridad las variables **más importantes** para el modelo (Lección 15.6).
</details>

---
## 5. 📖 La deriva de concepto: la que no se ve en los datos

Ahora el caso peligroso. Simularemos **12 meses** de producción del modelo de cancelación. A partir del **mes 7**, un competidor empieza a pagar la penalización a quien rompa su contrato anual: **el contrato anual deja de retener a los clientes**. Las entradas siguen viéndose exactamente igual.

▶️ Primero entrenamos el modelo con los datos de referencia (como en la Lección 15.1):

In [ ]:
NUMERICAS = ["meses_cliente", "llamadas_soporte", "cargo_mensual", "contrato_anual"]
modelo = make_pipeline(
    ColumnTransformer([("num", StandardScaler(), NUMERICAS),
                       ("cat", OneHotEncoder(handle_unknown="ignore"), ["plan"])]),
    LogisticRegression(max_iter=1000))
modelo.fit(referencia.drop(columns="cancelo"), referencia.cancelo)

meses = []
for mes in range(1, 13):
    efecto = -1.8 if mes < 7 else 0.0                          # desde el mes 7 el contrato ya no retiene
    datos = generar_clientes(2_000, semilla=100 + mes, efecto_contrato=efecto)
    p = modelo.predict_proba(datos.drop(columns="cancelo"))[:, 1]
    meses.append({"mes": mes,
                  "PSI contrato_anual": psi(referencia.contrato_anual, datos.contrato_anual),
                  "PSI llamadas": psi(referencia.llamadas_soporte, datos.llamadas_soporte),
                  "prob. media predicha": p.mean(),
                  "tasa real de cancelación": datos.cancelo.mean(),
                  "AUC": roc_auc_score(datos.cancelo, p)})
tablero = pd.DataFrame(meses).set_index("mes").round(3)
tablero

### ✋ Ejercicio 7 — Lee el tablero
1. ¿Las columnas de **PSI** dan alguna alarma en el mes 7?
2. ¿Qué pasa con el **AUC** y con la **tasa real de cancelación** desde el mes 7?
3. ¿Qué columna **no** se podría calcular el mismo día en que el modelo hace sus predicciones? ¿Por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No**: los PSI siguen prácticamente en 0. Las entradas no cambiaron en nada.
2. La **tasa real de cancelación** salta (de ≈ 0.35 a ≈ 0.49) y el **AUC** cae (de ≈ 0.86 a ≈ 0.79). La **probabilidad media predicha** no se mueve (≈ 0.35): el modelo sigue "tranquilo", subestimando las cancelaciones.
3. El **AUC** y la **tasa real**: necesitan saber quién canceló de verdad, y eso se sabe **semanas o meses después** (retraso de las etiquetas, *label delay*).

Esta es la lección más importante del monitoreo: **la deriva de concepto no se ve mirando solo las entradas**. Hace falta monitorear el **desempeño** con etiquetas reales en cuanto lleguen, y mientras tanto vigilar señales indirectas (por ejemplo, que la probabilidad media predicha se aleje de la tasa real del último mes con etiquetas).
</details>

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(tablero.index, tablero["AUC"], "o-", c="tab:red", label="AUC (necesita etiquetas)")
ax1.set_ylabel("AUC"); ax1.set_xlabel("Mes")
ax2 = ax1.twinx()
ax2.plot(tablero.index, tablero["PSI contrato_anual"], "s--", c="tab:blue", label="PSI contrato_anual")
ax2.set_ylabel("PSI"); ax2.set_ylim(0, 0.3)
ax1.axvline(6.5, c="gray", ls=":"); ax1.set_title("Deriva de concepto: el PSI no ve nada, el AUC sí")
fig.legend(loc="lower left"); plt.show()

---
## 6. 📖 Perillas del monitoreo

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| Tamaño de la ventana actual | Cuántos datos se comparan cada vez | Menos ruido, pero detecta más tarde | Detecta antes, más falsas alarmas | 1 semana–1 mes (que tenga ≥ 1,000 casos) |
| Umbral de PSI | Cuándo se dispara la alerta | Menos alertas, se escapan cambios moderados | Más alertas | 0.1 (vigilar) · 0.25 (actuar) |
| Umbral del valor p | Ídem para pruebas estadísticas | — | Más estricto | 0.05 corregido por el número de pruebas |
| Umbral de caída de desempeño | Cuándo se reentrena | Toleras más pérdida | Reentrenas más seguido | Por ejemplo, AUC −0.03 respecto a la validación |
| Ventana de referencia | Contra qué se compara | Toda la historia: estable | Últimos meses: sigue cambios estacionales | Datos de entrenamiento del modelo vigente |

### ✏️ Ejercicio 8 — La ventana pequeña
Con ventanas **pequeñas** (pocos clientes por mes) el PSI se vuelve ruidoso aunque nada cambie. Prueba `N_CLIENTES` = 100, 1,000 y 10,000 en 12 meses **sin ninguna deriva**. ¿Cuántos meses superan PSI = 0.1 por puro azar?

✍️ *Mi predicción:*

In [ ]:
N_CLIENTES = 100     # ✏️ prueba 100, 1_000 y 10_000

psis = [psi(referencia.cargo_mensual, generar_clientes(N_CLIENTES, semilla=200 + m).cargo_mensual) for m in range(12)]
print(f"{N_CLIENTES:>6} clientes/mes → PSI de cada mes: {np.round(psis, 3)}")
print(f"Meses con PSI > 0.1 sin deriva real: {sum(p > 0.1 for p in psis)} de 12")

<details><summary><b>👀 Ver respuesta</b></summary>

- Con **100** clientes, el PSI de un mes sin deriva salta entre ≈ 0.03 y ≈ 0.32, y **5 de 12 meses** superan 0.1: falsas alarmas (¡uno incluso supera 0.25!).
- Con **1,000** baja a ≈ 0.01: ninguna alarma.
- Con **10,000**, prácticamente 0.

Los umbrales clásicos del PSI (0.1 / 0.25) suponen ventanas razonablemente grandes. Si tienes pocos casos, junta más tiempo en cada ventana.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Alertas todos los meses que nadie revisa | Valores p con muchos datos o muchas variables | PSI o tamaño de efecto, corrección por múltiples pruebas, priorizar variables importantes |
| El negocio se queja de que el modelo falla, pero el tablero está en verde | Deriva de concepto: solo se vigilan las entradas | Monitorear desempeño con etiquetas reales; comparar predicción media vs. realidad |
| Alarmas en diciembre de cada año | Estacionalidad | Comparar con el mismo período del año anterior |
| Una variable "deriva" de golpe a valores imposibles | Error en el sistema de datos, no un cambio del mundo | Validación de entradas (Lección 15.1); revisar el proceso que genera los datos |
| Se descubre la caída de desempeño 6 meses tarde | Etiquetas con mucho retraso y nadie las cruza | Automatizar el cruce de predicciones con resultados reales |

### 🔮 Ejercicio 9 — Diagnóstico
El PSI de `edad` en un modelo de seguros pasa de 0.02 a **2.8** de un día para otro. Al revisar, casi todos los clientes nuevos tienen edad = 0. ¿Es deriva de datos? ¿Qué hay que hacer?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

Casi seguro **no** es un cambio del mundo (no aparecieron miles de bebés asegurándose): es un **error en la tubería de datos** (un campo vacío que se rellena con 0, un cambio de formato de fecha…). No hay que reentrenar: hay que **arreglar los datos** y, mientras tanto, rechazar o marcar esas predicciones. Un PSI enorme y repentino suele ser un problema de **ingeniería**; una subida gradual suele ser el **mundo** cambiando.
</details>

---
## 8. 📋 Ficha: qué monitorear

| Capa | Qué vigilar | Herramienta | ¿Necesita etiquetas? |
|---|---|---|---|
| Datos de entrada | Vacíos, rangos, categorías nuevas | Validación (Lección 15.1) | No |
| Distribución de entradas | Deriva de datos | PSI, KS, χ² | No |
| Distribución de predicciones | Cambios en la probabilidad media o en el % de casos positivos | PSI de las predicciones | No |
| Desempeño | Deriva de concepto, caída real | AUC, precisión, recall, costo de negocio (Lección 4.8) | **Sí** |
| Equidad | Desempeño por grupo | Métricas por grupo (Lección 15.4) | Sí |
| Operación | Latencia, errores, volumen de peticiones | Herramientas de sistemas | No |

## 🧠 Resumen en 5 puntos
1. Los modelos envejecen porque el mundo cambia: **deriva de datos** (cambian las entradas), **de concepto** (cambia la relación) y **de proporción de clases**.
2. La deriva de datos se detecta comparando distribuciones: **KS** (numéricas), **χ²** (categóricas) y **PSI** (cuánto cambió).
3. Con muchos datos o muchas variables, los valores p **alarman por nada**: mira el tamaño del efecto y corrige por múltiples pruebas.
4. La deriva de concepto **no se ve en las entradas**: hay que medir el desempeño con etiquetas reales (que llegan con retraso).
5. Un buen tablero vigila entradas, predicciones, desempeño, equidad y operación, con umbrales que el equipo realmente atienda.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** Tras una campaña en redes, llegan muchos clientes más jóvenes que antes. Esto es…
- a) Deriva de concepto · b) Deriva de datos · c) Sobreajuste · d) Fuga de datos

**2.** Un PSI de 0.35 en una variable importante sugiere…
- a) Nada relevante · b) Vigilar · c) Un cambio grande: investigar y probablemente reentrenar · d) Que el modelo es perfecto

**3.** Con 5 millones de filas, KS da p = 10⁻¹² para una variable cuyo histograma se ve idéntico. ¿Qué haces?
- a) Reentrenar de inmediato · b) Mirar el tamaño del efecto (estadístico KS o PSI) antes de alarmarte · c) Borrar la variable · d) Bajar el umbral del valor p

**4.** ¿Cuál de estas señales **puede** revelar una deriva de concepto?
- a) El PSI de las entradas · b) La caída del AUC medida con etiquetas reales · c) La latencia del servidor · d) El tamaño del archivo del modelo

**5.** Monitoreas 100 variables con p < 0.05 sin corrección. Sin deriva real, ¿cuántas falsas alarmas esperas por mes?
- a) 0 · b) ~5 · c) ~50 · d) 100

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** 2. **c)** 3. **b)** 4. **b)** 5. **b)** (100 × 0.05)
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** En un modelo de crédito, la etiqueta "¿pagó el préstamo?" se conoce **12 meses** después. ¿Cómo monitorearías el desempeño mientras tanto?

**B. Caso de análisis 💼** Durante la pandemia, muchos modelos de demanda fallaron a la vez. Un gerente pregunta: *"¿Por qué el sistema de monitoreo no nos avisó antes?"*. Explícale, con el vocabulario de esta lección, qué se pudo haber visto y qué no.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Con señales **tempranas**: deriva de entradas (PSI), deriva de las predicciones (¿cambió el % de solicitudes con riesgo alto?), y etiquetas **intermedias** que llegan antes: atrasos de 30 o 60 días en el pago, que suelen anticipar el impago. Además, analizar cohortes: los préstamos de hace 12 meses ya tienen etiqueta definitiva.

**B.** Algo **sí** se pudo ver rápido: las entradas cambiaron de golpe (menos visitas a tiendas, más pedidos en línea), así que un monitoreo de deriva de datos habría dado alarmas en días. Lo que **no** se ve en las entradas es si la **relación** cambió (deriva de concepto); eso solo se confirma comparando predicciones con ventas reales, que llegan con retraso. Además, una alarma no basta: alguien debe tener un **plan de acción** (usar reglas manuales, reentrenar con datos recientes, ampliar márgenes de error) para cuando suena.
</details>

### 9.3 🧩 Práctica integradora: "El tablero de la aseguradora"

Una aseguradora tiene un modelo de cancelación de pólizas. Te entregan **9 meses** de datos de producción y quieren saber **qué cambió y cuándo**. Lo harás en 5 pasos cortos.

**Paso 1 ▶️ — Los datos de producción**

In [ ]:
def mes_de_produccion(mes):
    inflacion = 1.0 + 0.03 * max(0, mes - 3)                       # desde el mes 4, +3 % por mes
    prob_planes = (.5, .3, .2) if mes < 6 else (.3, .3, .4)        # desde el mes 6, promo premium
    efecto = -1.8 if mes < 8 else -0.6                             # desde el mes 8, el contrato retiene menos
    return generar_clientes(2_000, semilla=300 + mes, inflacion=inflacion,
                            efecto_contrato=efecto, prob_planes=prob_planes)

produccion = {mes: mes_de_produccion(mes) for mes in range(1, 10)}
print({mes: len(df) for mes, df in produccion.items()})

> 🙈 Haz como si no hubieras leído la función: el reto es **descubrir** los cambios con el tablero.

**Paso 2 ✏️ — Construye el tablero de entradas.** Completa la función para que calcule, para cada mes, el PSI de las tres numéricas principales y el valor p de χ² del `plan`.

In [ ]:
def fila_tablero(mes, datos):
    fila = {"mes": mes}
    for col in ["cargo_mensual", "meses_cliente", "llamadas_soporte"]:
        fila[f"PSI {col}"] = psi(referencia[col], datos[col])
    tabla_plan = pd.DataFrame({"ref": referencia.plan.value_counts(), "act": datos.plan.value_counts()}).fillna(0)
    fila["p χ² plan"] = chi2_contingency(tabla_plan.values)[1]
    return fila

tablero_entradas = pd.DataFrame([fila_tablero(m, d) for m, d in produccion.items()]).set_index("mes")
tablero_entradas.round(3)

✋ ¿Qué variables derivan y desde qué mes? ¿Alguna entra en la zona de "vigilar" (PSI > 0.1)?

✍️ *Mi lectura:*

<details><summary><b>👀 Ver respuesta</b></summary>

- `cargo_mensual`: PSI ≈ 0.01 en los meses 1–4, y luego crece cada mes: ≈ 0.05 en el mes 5, ≈ 0.09 en el 6, cruza 0.1 (**vigilar**) en el **mes 7** y llega a ≈ **0.21** en el mes 9, acercándose a "investigar". Es la inflación gradual, que empezó en el mes 4.
- `plan`: valor p de χ² grande en los meses 1–5 y diminuto desde el **mes 6**: cambió la mezcla de planes (la promoción).
- `meses_cliente` y `llamadas_soporte`: estables todo el tiempo.

Fíjate en que la inflación de 3 % mensual tarda varios meses en notarse: los cambios **graduales** son difíciles de detectar mes a mes.
</details>

**Paso 3 ▶️ — Añade el desempeño.** Llegaron las etiquetas reales. Calcula para cada mes el AUC, la probabilidad media predicha y la tasa real.

In [ ]:
desempeno = []
for mes, datos in produccion.items():
    p = modelo.predict_proba(datos.drop(columns="cancelo"))[:, 1]
    desempeno.append({"mes": mes, "AUC": roc_auc_score(datos.cancelo, p),
                      "prob. media predicha": p.mean(), "tasa real": datos.cancelo.mean()})
tablero_desempeno = pd.DataFrame(desempeno).set_index("mes")
tablero_desempeno.round(3)

✋ ¿En qué mes cae el desempeño? ¿Coincide con alguna alarma del tablero de entradas?

✍️ *Mi lectura:*

<details><summary><b>👀 Ver respuesta</b></summary>

El AUC se mantiene alrededor de **0.85–0.87** hasta el mes 7 y cae en el **mes 8** (≈ 0.83). Al mismo tiempo la **tasa real** sube (de ≈ 0.34–0.36 a ≈ 0.41–0.44) mientras la probabilidad media predicha apenas se mueve (≈ 0.38): el modelo **subestima** las cancelaciones.

**No coincide** con ninguna alarma nueva del tablero de entradas: en el mes 8 no cambió ninguna entrada de forma especial. Es **deriva de concepto** (el contrato retiene menos). Y al revés: la inflación y la promoción (meses 4–6) cambiaron las entradas **sin** dañar el desempeño. Moraleja: **deriva de datos ≠ modelo roto**, y **modelo roto ≠ deriva de datos**.
</details>

**Paso 4 ✏️ — ¿Qué variable explica el cambio?** Cuando sospechas deriva de concepto, compara la relación entre una variable y la etiqueta **antes y después**. Calcula la tasa de cancelación según `contrato_anual` en los meses 1–7 y 8–9.

In [ ]:
antes = pd.concat([produccion[m] for m in range(1, 8)])
despues = pd.concat([produccion[m] for m in range(8, 10)])
comparacion = pd.DataFrame({
    "meses 1–7": antes.groupby("contrato_anual").cancelo.mean(),
    "meses 8–9": despues.groupby("contrato_anual").cancelo.mean()})
comparacion.index = ["sin contrato anual", "con contrato anual"]
comparacion.round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

Los clientes **sin** contrato anual cancelan igual que antes (≈ 0.48), pero los clientes **con** contrato anual pasan de cancelar ≈ 0.21 a ≈ 0.37: casi el doble. El contrato ya no los retiene como antes. Esa es la relación que cambió, y el modelo (entrenado cuando el contrato retenía mucho) no puede saberlo.

Este análisis "por segmentos" es la forma práctica de **localizar** una deriva de concepto.
</details>

**Paso 5 ✋ — Informe para la gerencia.** Escribe 3–4 frases: qué cambió, cuándo, si afecta al modelo y qué recomiendas.

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> «Desde abril (mes 4) los cargos mensuales suben de forma gradual y desde junio (mes 6) hay muchos más clientes en el plan premium; ninguno de esos cambios ha afectado la calidad del modelo. El problema real empezó en agosto (mes 8): los clientes con contrato anual ahora cancelan casi el doble que antes, posiblemente por una oferta de la competencia, y el modelo subestima las cancelaciones porque aprendió que el contrato los retenía. Recomiendo reentrenar el modelo con datos recientes (Lección 15.3), revisar con el área comercial qué cambió en el mercado y añadir al tablero un indicador mensual de cancelación por tipo de contrato.»
</details>

---
## 📚 Para profundizar
- 📰 **Artículo:** Gama et al. (2014), *A survey on concept drift adaptation* (ACM Computing Surveys).
- 📘 **Libro:** Chip Huyen, *Designing Machine Learning Systems*, capítulo 8 (*Data Distribution Shifts and Monitoring*).
- 🛠️ **Herramientas (para Colab o tu empresa):** `evidently` y `nannyml` generan tableros de deriva automáticamente; `alibi-detect` incluye detectores más avanzados. NannyML incluso estima el desempeño **sin etiquetas** en algunos casos.
- 📰 **Lectura:** *Population Stability Index*: busca su uso en riesgo de crédito para ver cómo lo aplican los bancos.

---
⬅️ **Anterior:** [15.1 Del notebook a producción: MLOps en una página](15.1-del-notebook-a-produccion.ipynb) · ➡️ **Siguiente:** [15.3 Reentrenamiento y versionado de datos y modelos](15.3-reentrenamiento-y-versionado.ipynb)